# APIM Responses web search with Azure Functions

Deploy an Entra-only gateway that buffers the initial Foundry response, checks for a completed hosted `web_search_call`, and sends that response plus a CSV of URLs to an Azure Function. The Function uses a **GPT-5.6 Luna deployment on Foundry** with no tools to rewrite the existing answer. Its prompt allows source URLs only from the CSV domains and their subdomains, and requires removing all other (blocked-domain) URLs from the final answer. When `stream=true`, the Function relays final Responses SSE events through APIM as they arrive.

**Audience:** Developers working with APIM policies, Azure Functions, and Responses API.

**Prerequisites:** Python 3.12+, Azure CLI 2.66+, `az login`, Contributor plus RBAC Administrator (or Owner) in the lab and Foundry resource groups, permission to create Entra app registrations/service principals, and an existing Foundry deployment of `gpt-5.6-luna` supporting Responses and `web_search`. Tenant policy must permit this public-client interactive sign-in. Azure model availability is checked below; another model is never selected automatically.

From this directory run `uv sync --group dev` and select this lab's `.venv` kernel. Copy `.env.example` to `.env` and fill in the Foundry settings. Read [README.md](README.md) for the architecture and limitations.

Running deployment cells creates billable APIM, a B1 hosting plan, Function, storage, role assignments, and three secret-free app registrations. The existing Foundry model is reused. Requests that search normally make **two model requests and one search stage**. Cleanup has a separate notebook.


## 1. Configure and validate the existing model deployment
The endpoint is the account's **OpenAI v1 endpoint**, not a Foundry project URL. The spelling in the model catalog is `gpt-5.6-luna`; Azure deployment names can differ. All command arguments use subprocess lists, and tokens are never written to disk.


In [ ]:
import json
import os
import re
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from urllib.parse import urlsplit
from uuid import UUID

import httpx
import msal
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from lab_helpers import az, citations, create_lab_apps, iter_sse, output_text

LAB = Path.cwd()
assert (LAB / 'main.bicep').exists(), 'Use labs/responses-web-search-function as the working directory.'
load_dotenv(LAB / '.env')
sys.path.insert(0, str(LAB / 'src'))
from processing import parse_urls_csv

resource_group = os.getenv('LAB_RESOURCE_GROUP', 'lab-responses-web-search-function')
location = os.getenv('LAB_LOCATION', 'westus2')
apim_sku = os.getenv('LAB_APIM_SKU', 'Basicv2')
foundry_name = os.environ['FOUNDRY_NAME']
foundry_group = os.environ['FOUNDRY_RESOURCE_GROUP']
foundry_deployment = os.getenv('FOUNDRY_DEPLOYMENT', 'gpt-5.6-luna')
account = az('account', 'show')
tenant_id, subscription_id = account['tenantId'], account['id']
foundry_subscription = os.getenv('FOUNDRY_SUBSCRIPTION_ID', subscription_id)
user = az('ad', 'signed-in-user', 'show')
caller_object_id = str(UUID(user['id']))
publisher_email = os.getenv('LAB_PUBLISHER_EMAIL') or user.get('mail') or user['userPrincipalName']

foundry = az('cognitiveservices', 'account', 'show', '--name', foundry_name,
             '--resource-group', foundry_group, '--subscription', foundry_subscription)
subdomain = foundry['properties']['customSubDomainName']
foundry_endpoint = os.getenv('FOUNDRY_ENDPOINT', f'https://{subdomain}.openai.azure.com/openai/v1').rstrip('/')
parsed = urlsplit(foundry_endpoint)
assert parsed.scheme == 'https' and parsed.path == '/openai/v1'
assert not parsed.query and not parsed.fragment and not parsed.username and not parsed.password
assert parsed.port in (None, 443) and parsed.hostname.endswith(('.openai.azure.com', '.services.ai.azure.com'))
assert re.fullmatch(r'[A-Za-z0-9_.-]+', foundry_deployment), 'Deployment name contains unsupported policy characters.'
assert resource_group != foundry_group or subscription_id != foundry_subscription, 'Use a separate lab resource group for safe cleanup.'
models = az('cognitiveservices', 'account', 'deployment', 'list', '--name', foundry_name,
            '--resource-group', foundry_group, '--subscription', foundry_subscription)
selected = next((model for model in models if model['name'] == foundry_deployment), None)
assert selected, f'Create the {foundry_deployment!r} Foundry deployment before continuing.'
assert selected['properties']['model']['name'] == 'gpt-5.6-luna', 'The selected deployment must host gpt-5.6-luna.'
print(f'Tenant: {tenant_id}; subscription: {subscription_id}; lab resource group: {resource_group}')
print(f'Foundry deployment: {foundry_deployment}; region: {foundry["location"]}')


## 2. Run local behavior tests
These tests do not call Azure. They cover CSV parsing, hosted-tool semantics, managed-identity bearer authentication, JSON pass-through, genuine incremental SSE delivery, disconnect cleanup, upstream errors, and truncated streams. APIM policy execution and Easy Auth require the later live tests.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], check=True)


## 3. Create Entra registrations
Create separate Gateway API, notebook public client, and Function API registrations. The helper preauthorizes only this notebook client for the lab's scopes. It creates no client secrets. The Function scope exists so the negative test can acquire a valid Function-audience token; Easy Auth independently denies every principal except APIM.

The ignored `.lab-state.json` records IDs for retry and cleanup, with no credentials. If tenant policy prevents app creation or consent, an administrator must provision these lab registrations before proceeding; do not switch to API keys.


In [ ]:
state_path = LAB / '.lab-state.json'
state = json.loads(state_path.read_text()) if state_path.exists() else {'tenant_id': tenant_id}
assert state['tenant_id'] == tenant_id
if state.get('subscription_id'):
    assert state['subscription_id'] == subscription_id and state['resource_group'] == resource_group
state.update(subscription_id=subscription_id, resource_group=resource_group,
             foundry_subscription_id=foundry_subscription, foundry_id=foundry['id'],
             deployment_name='web-search-function')
# Persist cleanup coordinates before creating the first registration.
state_path.write_text(json.dumps(state, indent=2) + '\n')
state = create_lab_apps(state_path, tenant_id, resource_group)
gateway_client_id = state['gateway_app']['appId']
function_client_id = state['function_app']['appId']
print('Three Entra applications are ready; no secrets were created.')


## 4. Deploy infrastructure and the Function
APIM and the Function receive **Cognitive Services OpenAI User** on the existing Foundry resource. The HTTP-only Function host receives **Storage Blob Data Owner** on its own storage account, whose shared-key authentication is disabled. Easy Auth requires the tenant, Function audience, and APIM's managed-identity object ID.

APIM creation can take 30–60 minutes. Keep the notebook running. RBAC propagation may take several more minutes. This lab intentionally omits body logging and inherited outbound body transformations because they would buffer the final stream.


In [ ]:
az('group', 'create', '--name', resource_group, '--location', location)
params = {
    'location': location, 'apimSku': apim_sku, 'publisherEmail': publisher_email,
    'foundryName': foundry_name, 'foundryResourceGroup': foundry_group,
    'foundrySubscriptionId': foundry_subscription, 'foundryEndpoint': foundry_endpoint,
    'foundryDeployment': foundry_deployment, 'gatewayClientId': gateway_client_id,
    'functionClientId': function_client_id, 'callerObjectId': caller_object_id,
}
(LAB / 'params.json').write_text(json.dumps({
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0', 'parameters': {key: {'value': value} for key, value in params.items()},
}, indent=2))
print('Deploying Azure resources. This may take 30–60 minutes.')
deployment = az('deployment', 'group', 'create', '--name', 'web-search-function',
                '--resource-group', resource_group, '--template-file', 'main.bicep',
                '--parameters', '@params.json')
outputs = {key: item['value'] for key, item in deployment['properties']['outputs'].items()}
state['outputs'] = outputs
state_path.write_text(json.dumps(state, indent=2) + '\n')
gateway_url, function_url = outputs['gatewayUrl'], outputs['functionUrl']
print(gateway_url)


In [ ]:
# Package only deployable source; never include .env, tests, notebooks, or tokens.
with zipfile.ZipFile('function.zip', 'w', zipfile.ZIP_DEFLATED) as package:
    for filename in ('function_app.py', 'processing.py', 'host.json', 'requirements.txt'):
        package.write(LAB / 'src' / filename, filename)
print('Publishing with Entra authentication and an Azure remote Python build.')
_ = az('functionapp', 'deployment', 'source', 'config-zip', '--resource-group', resource_group,
       '--name', outputs['functionAppName'], '--src', 'function.zip', '--build-remote', 'true', '--timeout', '1200')
functions = az('functionapp', 'function', 'list', '--resource-group', resource_group,
               '--name', outputs['functionAppName'])
assert any(item['name'].endswith('/process') for item in functions), 'Function indexing is not ready. Inspect deployment logs and retry this cell.'
print('Function process is indexed.')


## 5. Verify keyless configuration
These management-plane assertions check platform authentication and role wiring. The live inference tests below verify that the managed identities actually work after role propagation.


In [ ]:
subscription_prefix = f'/subscriptions/{subscription_id}/resourceGroups/{resource_group}'
site_id = f'{subscription_prefix}/providers/Microsoft.Web/sites/{outputs["functionAppName"]}'
auth = az('rest', '--url', f'https://management.azure.com{site_id}/config/authsettingsV2?api-version=2024-04-01')['properties']
assert auth['platform']['enabled'] and auth['globalValidation']['requireAuthentication']
validation = auth['identityProviders']['azureActiveDirectory']['validation']
assert validation['defaultAuthorizationPolicy']['allowedPrincipals']['identities'] == [outputs['apimPrincipalId']]
assert function_client_id in validation['allowedAudiences']
for name in ('scm', 'ftp'):
    publishing = az('rest', '--url', f'https://management.azure.com{site_id}/basicPublishingCredentialsPolicies/{name}?api-version=2024-04-01')
    assert publishing['properties']['allow'] is False
storage = az('storage', 'account', 'show', '--resource-group', resource_group, '--name', outputs['storageName'])
assert storage['allowSharedKeyAccess'] is False
assert len(outputs['foundryRoleAssignmentIds']) == 2
print('Easy Auth, APIM identity allowlist, disabled publishing passwords, and disabled storage keys verified.')


## 6. Authenticate the notebook
Sign in as the same user returned by Azure CLI in step 1. Use the printed device-login link/code. MSAL caches tokens **only in kernel memory** and refreshes them before each request. If your tenant blocks device-code flow, use `acquire_token_interactive(scopes=scopes)` in a local browser instead.


In [ ]:
msal_client = msal.PublicClientApplication(state['notebook_app']['appId'],
                                         authority=f'https://login.microsoftonline.com/{tenant_id}')
def token_for(client_id):
    scopes = [f'api://{client_id}/access_as_user']
    accounts = msal_client.get_accounts()
    result = msal_client.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
    if not result or 'access_token' not in result:
        flow = msal_client.initiate_device_flow(scopes=scopes)
        if 'user_code' not in flow:
            raise RuntimeError(flow.get('error_description', 'Could not initiate sign-in'))
        print(flow['message'])
        result = msal_client.acquire_token_by_device_flow(flow)
    if 'access_token' not in result:
        raise RuntimeError(result.get('error_description', 'Sign-in failed'))
    return result['access_token']

def auth_headers():
    return {'Authorization': f'Bearer {token_for(gateway_client_id)}'}

headers = auth_headers()
print('Gateway token acquired and kept in memory.')


## 7. Test authentication and request validation
An unauthenticated or wrong-audience request must fail at APIM. A correctly-audienced Function token belonging to the user must fail with **403**, because only APIM's principal is allowed. The Function also rejects spoofed identity headers without a token. No model calls are made by these cases.


In [ ]:
base_request = {'model': foundry_deployment, 'input': 'Say hello.', 'max_output_tokens': 256}
with httpx.Client(timeout=30) as client:
    assert client.post(gateway_url, json=base_request).status_code == 401
    assert client.post(gateway_url, json=base_request, headers={'Authorization': 'Bearer invalid'}).status_code == 401
    function_token = token_for(function_client_id)
    assert client.post(gateway_url, json=base_request, headers={'Authorization': f'Bearer {function_token}'}).status_code == 401
    assert client.post(function_url, json={}, headers={'Authorization': f'Bearer {function_token}'}).status_code == 403
    assert client.post(function_url, json={}, headers={'x-ms-client-principal': 'spoofed'}).status_code == 401
    for invalid in (
        {'input': 'Test', 'tools': [{'type': 'web_search'}]},
        {'input': 'Test', 'stream': 'true'},
        {'input': 'Test', 'background': True},
        {'input': 'Test', 'tools': [{'type': 'function', 'name': 'web_search'}]},
    ):
        response = client.post(gateway_url, headers=auth_headers(), json=invalid)
        assert response.status_code == 400, response.text
print('Authentication, authorization, and pre-inference validation passed.')


## 8. Test JSON responses and routing
First verify the direct Foundry route. Then offer `web_search` but set `tool_choice=none` to verify that an unused tool does not trigger the Function. Finally require search and check that APIM returns the Function's new response ID, no additional search calls, and only allowed source URLs when citations are available.

The CSV supplies **allowed domains for the final-answer prompt**. For example, `https://learn.microsoft.com/azure/api-management/` permits source links on `learn.microsoft.com` and its subdomains; paths do not restrict the list to exact pages. All other domains are blocked. The first search uses the caller's tool configuration; the Function performs no search or URL fetch. URL removal depends on model instruction following. The checks below inspect URLs in answer text and annotations; the Function itself does not scrub the relayed response. When no allowed source supports a claim, the prompt requires an explanation of the evidence limitation without blocked URLs.


In [ ]:
urls_csv = (LAB / 'sample-urls.csv').read_text()
_, expected_domains = parse_urls_csv(urls_csv)
search_request = {
    'model': foundry_deployment,
    'input': 'Use web search to explain why Azure API Management must disable response buffering when forwarding SSE from an Azure Function. Cite official documentation.',
    'tools': [{'type': 'web_search'}], 'tool_choice': 'required',
    'reasoning': {'effort': 'low'}, 'max_output_tokens': 4096,
    'include': ['web_search_call.action.sources'], 'urls_csv': urls_csv,
}

def assert_final(document):
    assert document['status'] == 'completed', document.get('incomplete_details') or document.get('error')
    assert output_text(document)
    assert not any(item.get('type') == 'web_search_call' for item in document['output'])
    refs = citations(document)
    # With no hosted search, citations are ordinary Markdown links. Check all
    # visible HTTP(S) URLs as well as any structured citation annotations.
    for match in re.findall(r'https?://[^\s<>"`]+', output_text(document), flags=re.IGNORECASE):
        url = match.rstrip('.,;:!?)]}')
        if not any(ref['url'] == url for ref in refs):
            refs.append({'url': url, 'title': url})
    for ref in refs:
        hostname = (urlsplit(ref['url']).hostname or '').encode('idna').decode('ascii').lower().rstrip('.')
        assert any(hostname == domain or hostname.endswith('.' + domain) for domain in expected_domains), ref['url']
    # No citations is valid when the first response has no allowed evidence.
    return refs

with httpx.Client(timeout=240) as client:
    direct = client.post(gateway_url, headers=auth_headers(), json=base_request)
    direct.raise_for_status()
    assert direct.headers['x-lab-route'] == 'foundry'
    assert direct.headers['x-lab-initial-buffered'] == 'false'
    assert direct.json()['status'] == 'completed'
    no_search = client.post(gateway_url, headers=auth_headers(), json={**search_request, 'input': 'Say hello without searching.', 'tool_choice': 'none'})
    no_search.raise_for_status()
    assert no_search.headers['x-lab-route'] == 'foundry-buffered'
    assert not any(item['type'] == 'web_search_call' for item in no_search.json()['output'])
    final = client.post(gateway_url, headers=auth_headers(), json=search_request)
    final.raise_for_status()
    final_document = final.json()
    assert final.headers['x-lab-route'] == 'function'
    assert final.headers['x-lab-initial-buffered'] == 'true'
    assert final_document['id'] != final.headers['x-lab-initial-response-id']
refs = assert_final(final_document)
display(Markdown(output_text(final_document)))
# Always show clickable citations, including SDK annotation URLs.
import html
display(HTML('<ul>' + ''.join(f'<li><a href="{html.escape(ref["url"], quote=True)}">{html.escape(ref["title"])}</a></li>' for ref in refs) + '</ul>'))


## 9. Test streaming on every route
APIM consumes the entire first SSE stream when a search tool is offered. Only after its terminal response can APIM choose the Function. The first response's events are never mixed into the Function stream. The Function relays final upstream SSE, including deltas, citations, and `response.completed`; comment heartbeats may appear during long model reads.

Transport chunks are arbitrary. These tests parse complete SSE frames and compare final IDs. Timing is diagnostic rather than a flaky hard threshold. The local gated-stream test proves incremental Function delivery; the live delta/terminal timestamps help detect buffering introduced by your gateway or network.


In [ ]:
def run_stream(payload, expected_route):
    events, timestamps = [], []
    started = time.perf_counter()
    with httpx.Client(timeout=240) as client:
        with client.stream('POST', gateway_url, headers=auth_headers(), json={**payload, 'stream': True}) as response:
            response.raise_for_status()
            assert response.headers['content-type'].startswith('text/event-stream')
            assert response.headers['x-lab-route'] == expected_route
            first_id = response.headers.get('x-lab-initial-response-id')
            for event in iter_sse(response.iter_lines()):
                events.append(event)
                timestamps.append(time.perf_counter() - started)
                assert event['type'] not in ('error', 'response.failed', 'response.incomplete'), event
    completed = [event['response'] for event in events if event['type'] == 'response.completed']
    assert len(completed) == 1, 'Missing or duplicate completion.'
    final_response = completed[0]
    deltas = [index for index, event in enumerate(events) if event['type'] == 'response.output_text.delta']
    assert deltas, 'Expected incremental text events.'
    assert final_response['status'] == 'completed'
    if first_id:
        assert final_response['id'] != first_id
        # No object from the buffered first Responses stream should escape.
        assert all(event.get('response', {}).get('id') != first_id for event in events)
    print(f'{expected_route}: {len(events)} events; first text {timestamps[deltas[0]]:.2f}s; complete {timestamps[-1]:.2f}s')
    return final_response

direct_stream = run_stream(base_request, 'foundry')
unused_stream = run_stream({**search_request, 'input': 'Say hello without searching.', 'tool_choice': 'none'}, 'foundry-buffered')
final_stream = run_stream(search_request, 'function')
refs = assert_final(final_stream)
display(Markdown(output_text(final_stream)))
display(HTML('<ul>' + ''.join(f'<li><a href="{html.escape(ref["url"], quote=True)}">{html.escape(ref["title"])}</a></li>' for ref in refs) + '</ul>'))


## 10. Test Function validation through APIM
This negative case deliberately supplies an invalid CSV URL. It makes the **initial paid search call**, then proves that the Function rejects the CSV before making its own Foundry call. Production applications should validate their CSV before calling the gateway, as `parse_urls_csv` does above.

Offline tests cover upstream HTTP failures, timeouts, midstream failure, and disconnect cleanup without causing failures on the deployed service. Once an SSE response has started, an upstream failure is an SSE `error` event; clients must require `response.completed` rather than interpreting HTTP 200 alone as success.


In [ ]:
with httpx.Client(timeout=240) as client:
    invalid_csv = client.post(gateway_url, headers=auth_headers(), json={**search_request, 'urls_csv': 'https://127.0.0.1/private'})
    assert invalid_csv.status_code == 400, invalid_csv.text
    assert invalid_csv.headers['x-lab-route'] == 'function'
    assert invalid_csv.json()['error']['code'] == 'invalid_request'
print('Function CSV validation passed. Run clean-up-resources.ipynb when finished.')


## Troubleshooting and limits

- **Model/tool unavailable:** Confirm `gpt-5.6-luna`, Responses and hosted `web_search` for the initial call are available in your Foundry resource and region. This lab does not silently fall back to preview search or another model.
- **401/403:** Check the signed-in user's object ID, token audience and scope, Function Easy Auth allowlist, and the two Foundry RBAC assignments. After provisioning, allow time for role/token propagation and retry.
- **Function missing/502:** Inspect `az functionapp log deployment show`, the remote build, and Function indexing. Keep `PYTHON_ENABLE_INIT_INDEXING=1`. Never enable basic publishing credentials to work around an Entra deployment error.
- **Slow first event:** The first response must finish before the Function starts. The Function then starts a model request with no tools to rewrite the answer. APIM uses 180-second upstream timeouts; platform idle limits also apply. Keep this lab to short requests rather than long research runs.
- **Streaming buffered elsewhere:** Keep APIM outbound policies free of body reads, JSON transforms, and response-body diagnostics. `buffer-response=true` by itself buffers only chunks, not a complete SSE response; this lab explicitly reads the initial response body.
- **Scope:** Stateless text requests with only hosted `web_search`; no conversations, previous response IDs, background mode, or custom tool execution. The gateway caps output tokens at 4096, accepts requests up to 64 KiB, and rejects buffered responses over 1 MiB after reading them. These are lab limits, not a hard memory cap during upstream buffering.

See [README.md](README.md) for references and [clean-up-resources.ipynb](clean-up-resources.ipynb) to remove resources and registrations.
